# RACE Reading Comprehension — EDA & Training Notebook
**BS(CS) Spring 2026 | AL2002 Lab Project**

Run *Kernel → Restart & Run All* to execute end-to-end.

## Section 1 — Environment & Data Loading

In [ ]:
# ── Install / upgrade packages if needed (safe to re-run) ──
import subprocess, sys
pkgs = ['scikit-learn','gensim','xgboost','seaborn','matplotlib',
        'joblib','pandas','numpy','pytest','kagglehub']
subprocess.check_call([sys.executable,'-m','pip','install','--quiet',*pkgs])
print('All packages ready.')

In [ ]:
import os, re, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — safe for Colab & headless
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.sparse import hstack, save_npz, load_npz
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.semi_supervised import LabelPropagation
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, silhouette_score, r2_score, classification_report
)
warnings.filterwarnings('ignore')
np.random.seed(42)

IS_KAGGLE = os.path.exists('/kaggle/working')
BASE_DIR = '/kaggle/working/race_rc_project' if IS_KAGGLE else '..'
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
FIGURES_DIR = os.path.join(PROCESSED_DIR, 'figures')
MODELS_A_DIR = os.path.join(BASE_DIR, 'models', 'model_a', 'traditional')
MODELS_B_DIR = os.path.join(BASE_DIR, 'models', 'model_b', 'traditional')
for d in [RAW_DIR, FIGURES_DIR, PROCESSED_DIR, MODELS_A_DIR, MODELS_B_DIR]:
    os.makedirs(d, exist_ok=True)
print(f'Imports OK. Directories ready. BASE_DIR={BASE_DIR}')

### 1.1 Download dataset via kagglehub

In [ ]:
import kagglehub

# Download RACE dataset from Kaggle
path = kagglehub.dataset_download('ankitdhiman7/race-dataset')
print('Path to dataset files:', path)

# Locate the CSV file
csv_candidates = []
for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith('.csv'):
            csv_candidates.append(os.path.join(root, f))
print('Found CSVs:', csv_candidates)

In [ ]:
# Load and split the dataset 80-10-10
assert len(csv_candidates) > 0, 'No CSV found in downloaded dataset!'

# Use first CSV (or the largest one)
csv_path = sorted(csv_candidates, key=os.path.getsize)[-1]
print(f'Using: {csv_path}')

full_df = pd.read_csv(csv_path)
print('Full dataset shape:', full_df.shape)
print('Columns:', full_df.columns.tolist())
print(full_df.head(2))

In [ ]:
# Standardise column names (handle variations across Kaggle versions)
col_map = {}
for c in full_df.columns:
    cl = c.strip().lower()
    if cl in ('article','passage','context'): col_map[c] = 'article'
    elif cl == 'question': col_map[c] = 'question'
    elif cl in ('a','option_a','option a'): col_map[c] = 'A'
    elif cl in ('b','option_b','option b'): col_map[c] = 'B'
    elif cl in ('c','option_c','option c'): col_map[c] = 'C'
    elif cl in ('d','option_d','option d'): col_map[c] = 'D'
    elif cl in ('answer','label'): col_map[c] = 'answer'
    elif cl in ('id','example_id'): col_map[c] = 'id'
full_df.rename(columns=col_map, inplace=True)

# Keep only rows with all required columns
required = ['article','question','A','B','C','D','answer']
missing = [c for c in required if c not in full_df.columns]
if missing:
    print(f'WARNING: missing columns {missing}. Available: {full_df.columns.tolist()}')
else:
    full_df = full_df.dropna(subset=required).reset_index(drop=True)
    print(f'After dropna: {full_df.shape}')

In [ ]:
# 80-10-10 split
from sklearn.model_selection import train_test_split

n = len(full_df)
train_end = int(0.8 * n)
val_end   = int(0.9 * n)

full_df = full_df.sample(frac=1, random_state=42).reset_index(drop=True)
train_df = full_df.iloc[:train_end].reset_index(drop=True)
val_df   = full_df.iloc[train_end:val_end].reset_index(drop=True)
test_df  = full_df.iloc[val_end:].reset_index(drop=True)

print(f'Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}')

# Save raw splits for reference
train_df.to_csv(os.path.join(RAW_DIR, 'train.csv'), index=False)
val_df.to_csv(os.path.join(RAW_DIR, 'val.csv'),   index=False)
test_df.to_csv(os.path.join(RAW_DIR, 'test.csv'), index=False)
print(f'Splits saved to: {RAW_DIR}')

In [ ]:
# Sanity checks
for split_name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    assert all(c in df.columns for c in required), f'{split_name} missing columns'
    assert df[required].isnull().sum().sum() == 0, f'{split_name} has nulls'
    assert df['answer'].isin(['A','B','C','D']).all(), f'{split_name} has invalid answers'
    print(f'{split_name}: {len(df)} rows — OK')

## Section 2 — Exploratory Data Analysis

In [ ]:
# ── 2.1 Answer label distribution ──
fig, ax = plt.subplots(figsize=(6,4))
counts = train_df['answer'].value_counts().sort_index()
ax.bar(counts.index, counts.values, color=['#4C72B0','#DD8452','#55A868','#C44E52'])
ax.set_title('Answer Label Distribution (Train Set)')
ax.set_xlabel('Answer Option')
ax.set_ylabel('Count')
for i,(lbl,cnt) in enumerate(zip(counts.index, counts.values)):
    ax.text(i, cnt+50, str(cnt), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/01_answer_distribution.png', dpi=100)
plt.show()
print('Saved 01_answer_distribution.png')

In [ ]:
# ── 2.2 Article length distribution ──
train_df['article_len'] = train_df['article'].str.split().str.len()
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(train_df['article_len'], bins=50, color='steelblue', edgecolor='white')
ax.set_title('Article Length Distribution (word count)')
ax.set_xlabel('Words')
ax.set_ylabel('Frequency')
ax.axvline(train_df['article_len'].mean(), color='red', linestyle='--', label=f'Mean={train_df["article_len"].mean():.0f}')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/02_article_length.png', dpi=100)
plt.show()
print('Saved 02_article_length.png')

In [ ]:
# ── 2.3 Question length distribution ──
train_df['q_len'] = train_df['question'].str.split().str.len()
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(train_df['q_len'], bins=30, color='darkorange', edgecolor='white')
ax.set_title('Question Length Distribution (word count)')
ax.set_xlabel('Words')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/03_question_length.png', dpi=100)
plt.show()
print('Saved 03_question_length.png')

In [ ]:
# ── 2.4 Option length distribution (avg per option vs correct option) ──
for opt in ['A','B','C','D']:
    train_df[f'{opt}_len'] = train_df[opt].str.split().str.len()

avg_all = {opt: train_df[f'{opt}_len'].mean() for opt in ['A','B','C','D']}
correct_mask = {opt: train_df['answer']==opt for opt in ['A','B','C','D']}
avg_correct = {opt: train_df.loc[correct_mask[opt], f'{opt}_len'].mean() for opt in ['A','B','C','D']}

fig, ax = plt.subplots(figsize=(8,5))
x = np.arange(4)
w = 0.35
ax.bar(x - w/2, [avg_all[o] for o in 'ABCD'],   w, label='All options', color='steelblue')
ax.bar(x + w/2, [avg_correct[o] for o in 'ABCD'], w, label='Correct option', color='darkorange')
ax.set_xticks(x); ax.set_xticklabels(list('ABCD'))
ax.set_title('Average Option Length — All vs Correct')
ax.set_xlabel('Option'); ax.set_ylabel('Avg Word Count')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/04_option_length.png', dpi=100)
plt.show()
print('Saved 04_option_length.png')

In [ ]:
# ── 2.5 Question type breakdown (first word) ──
wh_words = ['who','what','where','when','why','how','which','whose']
def qtype(q):
    first = str(q).lower().split()[0].strip('\"\'')
    return first if first in wh_words else 'fill-in / other'

train_df['q_type'] = train_df['question'].apply(qtype)
qtc = train_df['q_type'].value_counts()

fig, ax = plt.subplots(figsize=(8,5))
ax.barh(qtc.index[::-1], qtc.values[::-1], color='mediumseagreen')
ax.set_title('Question Type Breakdown (First Word)')
ax.set_xlabel('Count')
for i,(idx,v) in enumerate(zip(qtc.index[::-1], qtc.values[::-1])):
    ax.text(v+10, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/05_question_types.png', dpi=100)
plt.show()
print('Saved 05_question_types.png')

In [ ]:
# ── 2.6 Middle vs High school split (from 'id' or difficulty column if present) ──
if 'id' in train_df.columns:
    train_df['difficulty'] = train_df['id'].astype(str).apply(
        lambda x: 'high' if 'high' in x.lower() else ('middle' if 'middle' in x.lower() else 'unknown')
    )
elif 'difficulty' not in train_df.columns:
    # Assign based on article length as proxy (shorter = middle school)
    med = train_df['article_len'].median()
    train_df['difficulty'] = train_df['article_len'].apply(lambda x: 'high' if x>=med else 'middle')

diff_counts = train_df['difficulty'].value_counts()
fig, ax = plt.subplots(figsize=(5,5))
ax.pie(diff_counts.values, labels=diff_counts.index, autopct='%1.1f%%',
       colors=['#4C72B0','#DD8452','#aec7e8'], startangle=140)
ax.set_title('Middle vs High School Split (Train)')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/06_difficulty_split.png', dpi=100)
plt.show()
print('Saved 06_difficulty_split.png')

In [ ]:
# ── 2.7 Answer label balance across splits ──
splits = {'train': train_df, 'val': val_df, 'test': test_df}
fig, axes = plt.subplots(1,3, figsize=(12,4), sharey=True)
for ax, (name, df) in zip(axes, splits.items()):
    c = df['answer'].value_counts().sort_index()
    ax.bar(c.index, c.values, color=['#4C72B0','#DD8452','#55A868','#C44E52'])
    ax.set_title(f'{name.capitalize()} ({len(df)} rows)')
    ax.set_xlabel('Answer')
    ax.set_ylabel('Count')
plt.suptitle('Answer Label Distribution by Split', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/07_answer_by_split.png', dpi=100)
plt.show()
print('Saved 07_answer_by_split.png')

In [ ]:
# ── 2.8 Summary statistics ──
all_tokens = ' '.join(train_df['article'].fillna('').tolist()).split()
vocab_size = len(set(all_tokens))
print('\n=== DATASET SUMMARY STATISTICS ===')
print(f"{'Metric':<35} {'Value'}")
print('-'*55)
print(f"{'Total questions (full)':<35} {len(full_df):,}")
print(f"{'Train / Val / Test':<35} {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")
print(f"{'Mean article length (words)':<35} {train_df['article_len'].mean():.1f}")
print(f"{'Median article length (words)':<35} {train_df['article_len'].median():.1f}")
print(f"{'Max article length (words)':<35} {train_df['article_len'].max()}")
print(f"{'Mean question length (words)':<35} {train_df['q_len'].mean():.1f}")
print(f"{'Train vocab size (unique tokens)':<35} {vocab_size:,}")
print(f"{'Answer A %':<35} {(train_df['answer']=='A').mean()*100:.1f}%")
print(f"{'Answer B %':<35} {(train_df['answer']=='B').mean()*100:.1f}%")
print(f"{'Answer C %':<35} {(train_df['answer']=='C').mean()*100:.1f}%")
print(f"{'Answer D %':<35} {(train_df['answer']=='D').mean()*100:.1f}%")

## Section 3 — Preprocessing Pipeline

In [ ]:
# ── 3.1 Text cleaning ──
import string

def clean_text(text):
    """Lowercase, remove punctuation, strip extra whitespace."""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply to all text columns
for df in [train_df, val_df, test_df]:
    df['article_clean']  = df['article'].apply(clean_text)
    df['question_clean'] = df['question'].apply(clean_text)
    for opt in ['A','B','C','D']:
        df[f'{opt}_clean'] = df[opt].apply(clean_text)

print('Text cleaning done.')
print('Example cleaned article:', train_df['article_clean'].iloc[0][:120])

In [ ]:
# ── 3.2 Data expansion: 1 row per option ──
def expand_df(df):
    rows = []
    for _, r in df.iterrows():
        for opt in ['A','B','C','D']:
            rows.append({
                'article'       : r['article_clean'],
                'question'      : r['question_clean'],
                'option'        : r[f'{opt}_clean'],
                'option_letter' : opt,
                'label'         : 1 if r['answer'] == opt else 0,
                'combined_text' : r['article_clean'] + ' [SEP] ' + r['question_clean'] + ' [SEP] ' + r[f'{opt}_clean'],
                'article_raw'   : r['article'],
                'question_raw'  : r['question'],
                'A_raw': r['A'], 'B_raw': r['B'], 'C_raw': r['C'], 'D_raw': r['D'],
                'answer'        : r['answer'],
            })
    return pd.DataFrame(rows)

print('Expanding datasets (1 row per option)...')
train_exp = expand_df(train_df)
val_exp   = expand_df(val_df)
test_exp  = expand_df(test_df)
print(f'Expanded → Train: {train_exp.shape}, Val: {val_exp.shape}, Test: {test_exp.shape}')
print(train_exp[['option_letter','label','combined_text']].head(4))

In [ ]:
# ── 3.3 One-Hot Encoding (CountVectorizer binary=True) ──
print('Building One-Hot vocabulary (top 5000 tokens)...')
ohe_vec = CountVectorizer(binary=True, max_features=5000, min_df=2)
X_ohe_train = ohe_vec.fit_transform(train_exp['combined_text'])
X_ohe_val   = ohe_vec.transform(val_exp['combined_text'])
X_ohe_test  = ohe_vec.transform(test_exp['combined_text'])
print(f'OHE feature matrix shapes: train={X_ohe_train.shape}, val={X_ohe_val.shape}, test={X_ohe_test.shape}')
joblib.dump(ohe_vec, f'{MODELS_A_DIR}/ohe_vectorizer.pkl')
print('OHE vectorizer saved.')

In [ ]:
# ── 3.4 Cosine similarity feature (article vs option) ──
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
from scipy.sparse import csr_matrix

print('Computing cosine similarity features...')
art_vec = CountVectorizer(binary=True, max_features=5000, vocabulary=ohe_vec.vocabulary_)

def cos_feature(df_exp):
    art_mat = art_vec.transform(df_exp['article'])
    opt_mat = art_vec.transform(df_exp['option'])
    # Row-wise cosine similarity
    scores = np.array([
        cos_sim(art_mat[i], opt_mat[i])[0,0]
        for i in range(art_mat.shape[0])
    ]).reshape(-1,1)
    return csr_matrix(scores)

cos_train = cos_feature(train_exp)
cos_val   = cos_feature(val_exp)
cos_test  = cos_feature(test_exp)
print('Cosine features computed.')

In [ ]:
# ── 3.5 Lexical features ──
def lexical_features(df_exp):
    feats = []
    for _, r in df_exp.iterrows():
        art_toks = set(r['article'].split())
        q_toks   = set(r['question'].split())
        opt_toks = set(r['option'].split())
        opt_list = r['option'].split()
        art_str  = r['article']

        option_length    = len(opt_list)
        question_length  = len(r['question'].split())
        keyword_overlap  = len(q_toks & opt_toks)
        opt_in_article   = len(opt_toks & art_toks)
        # position of first option token in article (normalised)
        pos = 0.0
        if opt_list and opt_list[0] in art_str:
            idx = art_str.find(opt_list[0])
            pos = idx / max(len(art_str), 1)
        feats.append([option_length, question_length, keyword_overlap, opt_in_article, pos])
    return csr_matrix(np.array(feats, dtype=np.float32))

print('Computing lexical features (this may take a minute)...')
lex_train = lexical_features(train_exp)
lex_val   = lexical_features(val_exp)
lex_test  = lexical_features(test_exp)
print('Lexical features computed.')

In [ ]:
# ── 3.6 Final feature matrix: hstack OHE + cosine + lexical ──
X_train = hstack([X_ohe_train, cos_train, lex_train])
X_val   = hstack([X_ohe_val,   cos_val,   lex_val])
X_test  = hstack([X_ohe_test,  cos_test,  lex_test])

y_train = train_exp['label'].values
y_val   = val_exp['label'].values
y_test  = test_exp['label'].values

print(f'Final feature shapes: train={X_train.shape}, val={X_val.shape}, test={X_test.shape}')
print(f'Label distributions — train: {Counter(y_train)}, val: {Counter(y_val)}, test: {Counter(y_test)}')

# Save feature matrices
save_npz(f'{PROCESSED_DIR}/X_train.npz', X_train)
save_npz(f'{PROCESSED_DIR}/X_val.npz',   X_val)
save_npz(f'{PROCESSED_DIR}/X_test.npz',  X_test)
np.save(f'{PROCESSED_DIR}/y_train.npy', y_train)
np.save(f'{PROCESSED_DIR}/y_val.npy',   y_val)
np.save(f'{PROCESSED_DIR}/y_test.npy',  y_test)

# Also save expanded dataframes for Model B and inference
train_exp.to_csv(f'{PROCESSED_DIR}/train_exp.csv', index=False)
val_exp.to_csv(f'{PROCESSED_DIR}/val_exp.csv',     index=False)
test_exp.to_csv(f'{PROCESSED_DIR}/test_exp.csv',   index=False)
print('Feature matrices and expanded dataframes saved.')

In [ ]:
%%writefile ../src/preprocessing.py
"""preprocessing.py — RACE dataset loading and feature engineering."""
import os, re, string
import numpy as np
import pandas as pd
import joblib
from scipy.sparse import hstack, load_npz, csr_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim


def clean_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', text).strip()


def expand_df(df):
    rows = []
    for _, r in df.iterrows():
        for opt in ['A','B','C','D']:
            a_c = clean_text(r['article'])
            q_c = clean_text(r['question'])
            o_c = clean_text(r[opt])
            rows.append({
                'article': a_c, 'question': q_c, 'option': o_c,
                'option_letter': opt,
                'label': 1 if r['answer'] == opt else 0,
                'combined_text': f'{a_c} [SEP] {q_c} [SEP] {o_c}',
                'article_raw': r['article'], 'question_raw': r['question'],
                'A_raw': r['A'], 'B_raw': r['B'], 'C_raw': r['C'], 'D_raw': r['D'],
                'answer': r['answer'],
            })
    return pd.DataFrame(rows)


def build_features(train_exp, val_exp, test_exp, save_dir='data/processed',
                   models_dir='models/model_a/traditional', max_features=5000):
    os.makedirs(save_dir, exist_ok=True)
    ohe_vec = CountVectorizer(binary=True, max_features=max_features, min_df=2)
    X_ohe_tr = ohe_vec.fit_transform(train_exp['combined_text'])
    X_ohe_va = ohe_vec.transform(val_exp['combined_text'])
    X_ohe_te = ohe_vec.transform(test_exp['combined_text'])
    joblib.dump(ohe_vec, os.path.join(models_dir, 'ohe_vectorizer.pkl'))

    art_vec = CountVectorizer(binary=True, vocabulary=ohe_vec.vocabulary_)
    def cos_feature(df_e):
        am = art_vec.transform(df_e['article'])
        om = art_vec.transform(df_e['option'])
        scores = np.array([cos_sim(am[i], om[i])[0,0] for i in range(am.shape[0])]).reshape(-1,1)
        return csr_matrix(scores)
    def lex_feat(df_e):
        feats = []
        for _, r in df_e.iterrows():
            at = set(r['article'].split()); qt = set(r['question'].split()); ot = set(r['option'].split())
            ol = list(r['option'].split()); art_s = r['article']
            pos = 0.0
            if ol and ol[0] in art_s: pos = art_s.find(ol[0]) / max(len(art_s),1)
            feats.append([len(ol), len(r['question'].split()), len(qt&ot), len(ot&at), pos])
        return csr_matrix(np.array(feats, dtype=np.float32))

    from scipy.sparse import save_npz
    for name, df_e, X_ohe in [('train',train_exp,X_ohe_tr),('val',val_exp,X_ohe_va),('test',test_exp,X_ohe_te)]:
        X = hstack([X_ohe, cos_feature(df_e), lex_feat(df_e)])
        save_npz(os.path.join(save_dir, f'X_{name}.npz'), X)
        np.save(os.path.join(save_dir, f'y_{name}.npy'), df_e['label'].values)
    return ohe_vec


def load_features(processed_dir='data/processed'):
    X_tr = load_npz(os.path.join(processed_dir, 'X_train.npz'))
    X_va = load_npz(os.path.join(processed_dir, 'X_val.npz'))
    X_te = load_npz(os.path.join(processed_dir, 'X_test.npz'))
    y_tr = np.load(os.path.join(processed_dir, 'y_train.npy'))
    y_va = np.load(os.path.join(processed_dir, 'y_val.npy'))
    y_te = np.load(os.path.join(processed_dir, 'y_test.npy'))
    return X_tr, X_va, X_te, y_tr, y_va, y_te


## Section 4 — Model A: Traditional ML (Answer Verification)

### 4a. Logistic Regression
Superclassifier over full OHE + cosine + lexical features. Strong baseline for text classification.

In [ ]:
# ── Helper: Exact Match accuracy for MC (4-way) ──
def exact_match(y_true_exp, y_pred_exp, n_options=4):
    """Fraction of questions where argmax predicted score == correct option."""
    n = len(y_true_exp) // n_options
    correct = 0
    for i in range(n):
        block = y_pred_exp[i*n_options:(i+1)*n_options]
        true_block = y_true_exp[i*n_options:(i+1)*n_options]
        if true_block.sum() == 0: continue
        if np.argmax(block) == np.argmax(true_block):
            correct += 1
    return correct / max(n, 1)

results = {}  # store results for comparison table
print('Helper defined.')

In [ ]:
t0 = time.time()
lr_model = LogisticRegression(max_iter=1000, C=1.0, solver='saga', n_jobs=-1)
lr_model.fit(X_train, y_train)
train_time_lr = time.time() - t0

lr_val_pred  = lr_model.predict(X_val)
lr_val_proba = lr_model.predict_proba(X_val)[:,1]

lr_acc = accuracy_score(y_val, lr_val_pred)
lr_f1  = f1_score(y_val, lr_val_pred, average='macro')
lr_em  = exact_match(y_val, lr_val_proba)

print(f'Logistic Regression — Val Accuracy: {lr_acc:.4f}, Macro F1: {lr_f1:.4f}, EM: {lr_em:.4f}')
print(f'Train time: {train_time_lr:.1f}s')
results['Logistic Regression'] = {'accuracy': lr_acc, 'f1': lr_f1, 'em': lr_em, 'time': train_time_lr}

joblib.dump(lr_model, f'{MODELS_A_DIR}/lr_model.pkl')
print('LR model saved.')

### 4b. Support Vector Machine (LinearSVC + Calibration)

In [ ]:
t0 = time.time()
svm_base  = LinearSVC(max_iter=2000, C=1.0)
svm_model = CalibratedClassifierCV(svm_base, cv=3)
svm_model.fit(X_train, y_train)
train_time_svm = time.time() - t0

svm_val_pred  = svm_model.predict(X_val)
svm_val_proba = svm_model.predict_proba(X_val)[:,1]

svm_acc = accuracy_score(y_val, svm_val_pred)
svm_f1  = f1_score(y_val, svm_val_pred, average='macro')
svm_em  = exact_match(y_val, svm_val_proba)

print(f'SVM (Calibrated) — Val Accuracy: {svm_acc:.4f}, Macro F1: {svm_f1:.4f}, EM: {svm_em:.4f}')
print(f'Train time: {train_time_svm:.1f}s')
results['SVM (Calibrated)'] = {'accuracy': svm_acc, 'f1': svm_f1, 'em': svm_em, 'time': train_time_svm}

joblib.dump(svm_model, f'{MODELS_A_DIR}/svm_model.pkl')
print('SVM model saved.')

### 4c. Naive Bayes — Question Type Classifier
Used to classify questions by type (Who/What/Where/…), a useful auxiliary task.

In [ ]:
# Prepare question-type classification data
q_vec = CountVectorizer(max_features=3000)
X_qtrain = q_vec.fit_transform(train_df['question_clean'])
X_qval   = q_vec.transform(val_df['question_clean'])

y_qtrain = train_df['q_type'].values
y_qval   = val_df['q_type'].apply(qtype).values if 'q_type' not in val_df.columns else val_df['q_type'].values

t0 = time.time()
nb_model = MultinomialNB()
nb_model.fit(X_qtrain, y_qtrain)
train_time_nb = time.time() - t0

nb_pred = nb_model.predict(X_qval)
nb_acc  = accuracy_score(y_qval, nb_pred)
nb_f1   = f1_score(y_qval, nb_pred, average='macro', zero_division=0)

print(f'Naive Bayes (Q-type) — Val Accuracy: {nb_acc:.4f}, Macro F1: {nb_f1:.4f}')
print(f'Train time: {train_time_nb:.1f}s')
results['Naive Bayes (Q-type)'] = {'accuracy': nb_acc, 'f1': nb_f1, 'em': 'N/A', 'time': train_time_nb}

joblib.dump(nb_model, f'{MODELS_A_DIR}/nb_model.pkl')
joblib.dump(q_vec,    f'{MODELS_A_DIR}/nb_vectorizer.pkl')
print('NB model saved.')

### 4d. Random Forest — Lexical Features Only

In [ ]:
# Extract lexical-only features (dense, suitable for RF)
# lexical: cols are [option_length, question_length, keyword_overlap, opt_in_article, answer_position]
lex_start = X_ohe_train.shape[1] + 1  # after OHE + cosine col
X_lex_train = np.asarray(lex_train.todense())
X_lex_val   = np.asarray(lex_val.todense())
X_lex_test  = np.asarray(lex_test.todense())

t0 = time.time()
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_lex_train, y_train)
train_time_rf = time.time() - t0

rf_val_pred  = rf_model.predict(X_lex_val)
rf_val_proba = rf_model.predict_proba(X_lex_val)[:,1]

rf_acc = accuracy_score(y_val, rf_val_pred)
rf_f1  = f1_score(y_val, rf_val_pred, average='macro')
rf_em  = exact_match(y_val, rf_val_proba)

print(f'Random Forest — Val Accuracy: {rf_acc:.4f}, Macro F1: {rf_f1:.4f}, EM: {rf_em:.4f}')
print(f'Train time: {train_time_rf:.1f}s')
results['Random Forest'] = {'accuracy': rf_acc, 'f1': rf_f1, 'em': rf_em, 'time': train_time_rf}

joblib.dump(rf_model, f'{MODELS_A_DIR}/rf_model.pkl')
print('RF model saved.')

In [ ]:
# ── 4e. Model A comparison table ──
print(f"{'Model':<25} | {'Accuracy':>8} | {'Macro F1':>8} | {'Exact Match':>11} | {'Train time':>10}")
print('-'*75)
for model_name, m in results.items():
    em_str = f"{m['em']:.4f}" if isinstance(m['em'], float) else m['em']
    print(f"{model_name:<25} | {m['accuracy']:>8.4f} | {m['f1']:>8.4f} | {em_str:>11} | {m['time']:>8.1f}s")

## Section 5 — Model A: Unsupervised & Semi-Supervised

### 5a. K-Means Clustering with Elbow Method

In [ ]:
# Work on a sample (K-Means on full sparse matrix is slow)
SAMPLE = min(10000, X_train.shape[0])
idx = np.random.choice(X_train.shape[0], SAMPLE, replace=False)
X_sample = X_train[idx].toarray()
y_sample = y_train[idx]

# Elbow method
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=5, max_iter=100)
    km.fit(X_sample)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(list(K_range), inertias, 'o-', color='steelblue')
ax.set_title('K-Means Elbow Curve')
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Inertia')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/08_kmeans_elbow.png', dpi=100)
plt.show()
print('Saved 08_kmeans_elbow.png')

In [ ]:
# Fit final K-Means (K=2 is natural: correct/incorrect)
BEST_K = 2
km_final = KMeans(n_clusters=BEST_K, random_state=42, n_init=10, max_iter=200)
km_labels = km_final.fit_predict(X_sample)

# Silhouette score (subsample for speed)
sil_idx = np.random.choice(len(km_labels), min(2000, len(km_labels)), replace=False)
sil_score = silhouette_score(X_sample[sil_idx], km_labels[sil_idx])

# Clustering purity
def purity_score(y_true, y_pred):
    from scipy.stats import mode
    total = 0
    for cluster in np.unique(y_pred):
        mask = y_pred == cluster
        majority = mode(y_true[mask], keepdims=True).count[0]
        total += majority
    return total / len(y_true)

purity = purity_score(y_sample, km_labels)
print(f'K-Means (K={BEST_K}) — Silhouette Score: {sil_score:.4f}, Purity: {purity:.4f}')

# PCA 2D visualization
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_sample[:2000])
fig, ax = plt.subplots(figsize=(7,5))
for c, label in [(0,'Cluster 0'),(1,'Cluster 1')]:
    mask = km_labels[:2000] == c
    ax.scatter(X_2d[mask,0], X_2d[mask,1], s=5, alpha=0.5, label=label)
ax.set_title('K-Means Clusters (PCA 2D)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/09_kmeans_pca.png', dpi=100)
plt.show()
print('Saved 09_kmeans_pca.png')

### 5b. Gaussian Mixture Model

In [ ]:
gmm = GaussianMixture(n_components=BEST_K, random_state=42, covariance_type='diag')
gmm.fit(X_sample)
gmm_labels_hard = gmm.predict(X_sample)
gmm_labels_soft  = gmm.predict_proba(X_sample)

sil_gmm = silhouette_score(X_sample[sil_idx], gmm_labels_hard[sil_idx])
log_lik = gmm.score(X_sample)
purity_gmm = purity_score(y_sample, gmm_labels_hard)

print(f'GMM — Silhouette: {sil_gmm:.4f}, Log-Likelihood: {log_lik:.4f}, Purity: {purity_gmm:.4f}')

print('\n--- Hard vs Soft Assignment Comparison (first 5 rows) ---')
for i in range(5):
    print(f'  Hard: {gmm_labels_hard[i]}  |  Soft: [{gmm_labels_soft[i,0]:.3f}, {gmm_labels_soft[i,1]:.3f}]  |  True: {y_sample[i]}')

### 5c. Label Propagation (Semi-Supervised)

In [ ]:
# Use 10% as labeled, rest as unlabeled (label=-1)
SS_SAMPLE = min(5000, X_sample.shape[0])
X_ss = X_sample[:SS_SAMPLE]
y_ss = y_sample[:SS_SAMPLE].copy()

labeled_idx = np.random.choice(SS_SAMPLE, int(0.1*SS_SAMPLE), replace=False)
y_ss_partial = np.full(SS_SAMPLE, -1)
y_ss_partial[labeled_idx] = y_ss[labeled_idx]

lp_model = LabelPropagation(kernel='knn', n_neighbors=7, max_iter=200)
lp_model.fit(X_ss, y_ss_partial)
lp_preds = lp_model.predict(X_ss)

# Evaluate only on unlabeled portion
unlabeled_idx = np.where(y_ss_partial == -1)[0]
lp_f1 = f1_score(y_ss[unlabeled_idx], lp_preds[unlabeled_idx], average='macro', zero_division=0)
print(f'Label Propagation — Unlabeled F1: {lp_f1:.4f}')

# Compare supervised LR F1 vs LP F1
lr_sample_f1 = f1_score(y_ss, lr_model.predict(csr_matrix(X_ss)), average='macro', zero_division=0)
print(f'\nComparison on same {SS_SAMPLE} samples:')
print(f"  Supervised LR Macro F1:    {lr_sample_f1:.4f}")
print(f"  Label Propagation Macro F1: {lp_f1:.4f} (unlabeled only)")

### 5d. Ensemble — Soft Voting (LR + SVM + RF)

In [ ]:
# Soft voting: average predicted probabilities from LR, SVM (calibrated), RF
# RF uses lexical features only — align it with the full val set
print('Computing ensemble predictions on validation set...')

lr_proba_val  = lr_model.predict_proba(X_val)[:,1]
svm_proba_val = svm_model.predict_proba(X_val)[:,1]
rf_proba_val  = rf_model.predict_proba(X_lex_val)[:,1]

ens_proba_val = (lr_proba_val + svm_proba_val + rf_proba_val) / 3
ens_pred_val  = (ens_proba_val >= 0.5).astype(int)

ens_acc = accuracy_score(y_val, ens_pred_val)
ens_f1  = f1_score(y_val, ens_pred_val, average='macro')
ens_em  = exact_match(y_val, ens_proba_val)

print(f'Ensemble (Soft Vote) — Val Accuracy: {ens_acc:.4f}, Macro F1: {ens_f1:.4f}, EM: {ens_em:.4f}')
results['Ensemble (Soft Vote)'] = {'accuracy': ens_acc, 'f1': ens_f1, 'em': ens_em, 'time': 0}

np.save(f'{PROCESSED_DIR}/ensemble_val_preds.npy', ens_proba_val)
print('Ensemble val predictions saved.')

## Section 6 — Model A: Template-Based Question Generation

In [ ]:
# ── 6a. Candidate sentence extraction by keyword overlap ──
def score_sentences(article, answer, top_k=3):
    sentences = [s.strip() for s in re.split(r'[.!?]', article) if len(s.strip()) > 10]
    ans_toks = set(clean_text(answer).split())
    scored = []
    for s in sentences:
        s_toks = set(clean_text(s).split())
        overlap = len(s_toks & ans_toks) / max(len(ans_toks), 1)
        scored.append((overlap, s))
    scored.sort(reverse=True)
    return [s for _, s in scored[:top_k]]

# Test
sample_row = train_df.iloc[0]
cands = score_sentences(sample_row['article'], sample_row[sample_row['answer']])
print('Top-3 candidate sentences:')
for i, c in enumerate(cands, 1):
    print(f'  {i}. {c[:120]}')

In [ ]:
# ── 6b. Wh-word template application ──
WH_TEMPLATES = [
    (r'^(\w+)\s+(.+)',          'What did {} do?'),
    (r'[A-Z][a-z]+(?:\s[A-Z][a-z]+)*', 'Who is ___?'),
]

def generate_question(sentence, answer):
    candidates = []
    # Fill-in-the-blank template
    ans_clean = clean_text(answer)
    if ans_clean in clean_text(sentence):
        blanked = re.sub(re.escape(ans_clean), '___', clean_text(sentence), count=1)
        candidates.append(f'Fill in the blank: "{blanked}"')
    # Capitalized token → Who template
    caps = re.findall(r'\b[A-Z][a-z]{2,}\b', sentence)
    if caps:
        candidates.append(f'Who or what is {caps[0]}?')
    # Generic What template
    words = sentence.split()
    if len(words) >= 4:
        subj = ' '.join(words[:2])
        candidates.append(f'What can be said about {subj}?')
    return candidates

sample_q_cands = generate_question(cands[0], sample_row[sample_row['answer']])
print('Generated question candidates:')
for q in sample_q_cands:
    print(f'  - {q}')

In [ ]:
# ── 6c. Question ranker (LinearSVC on candidate features) ──
def candidate_features(question_text, article, answer):
    q_len = len(question_text.split())
    starts_wh = int(question_text.lower().split()[0] in ['who','what','where','when','why','how','which'] if question_text.split() else 0)
    q_toks = set(clean_text(question_text).split())
    art_toks = set(clean_text(article).split())
    ans_toks = set(clean_text(answer).split())
    overlap_art = len(q_toks & art_toks) / max(len(q_toks), 1)
    overlap_ans = len(q_toks & ans_toks) / max(len(q_toks), 1)
    return [q_len, starts_wh, overlap_art, overlap_ans]

# Build training set: positive = RACE questions, negative = garbled sentences
pos_rows = train_df.sample(min(1000, len(train_df)), random_state=42)
X_qrank, y_qrank = [], []
for _, r in pos_rows.iterrows():
    ans = r[r['answer']]
    X_qrank.append(candidate_features(r['question'], r['article'], ans))
    y_qrank.append(1)
    # Negative: garbled sentence (just reversed words of a candidate sentence)
    garbled = ' '.join(r['article'].split()[:8][::-1]) + '?'
    X_qrank.append(candidate_features(garbled, r['article'], ans))
    y_qrank.append(0)

X_qrank = np.array(X_qrank); y_qrank = np.array(y_qrank)
q_ranker = LinearSVC(max_iter=1000)
q_ranker.fit(X_qrank, y_qrank)
print(f'Question ranker trained. Train acc: {accuracy_score(y_qrank, q_ranker.predict(X_qrank)):.4f}')
joblib.dump(q_ranker, f'{MODELS_A_DIR}/q_ranker.pkl')

In [ ]:
# ── 6d. Show 5 example (article snippet, generated Q, gold Q) triples ──
print('\n=== Example Generated vs Gold Questions ===')
for i in range(min(5, len(train_df))):
    r = train_df.iloc[i]
    ans = r[r['answer']]
    sents = score_sentences(r['article'], ans, top_k=1)
    if not sents: continue
    gen_qs = generate_question(sents[0], ans)
    if not gen_qs: continue
    # Rank
    feats = [candidate_features(q, r['article'], ans) for q in gen_qs]
    scores = q_ranker.decision_function(feats)
    best_q = gen_qs[np.argmax(scores)]
    print(f"\nArticle: ...{r['article'][:80]}...")
    print(f"Generated Q : {best_q}")
    print(f"Gold Q      : {r['question']}")

## Section 7 — Model B: Distractor & Hint Generation

### 7a–7c. Distractor Ranker

In [ ]:
# ── 7a. Candidate extraction: n-grams from article ──
def extract_candidates(article, answer, max_ngram=3):
    tokens = clean_text(article).split()
    ans_clean = clean_text(answer)
    candidates = set()
    for n in range(1, max_ngram+1):
        for i in range(len(tokens)-n+1):
            phrase = ' '.join(tokens[i:i+n])
            if phrase != ans_clean and len(phrase) > 2:
                candidates.add(phrase)
    return list(candidates)

# Test
r0 = train_df.iloc[0]
a0 = r0[r0['answer']]
cands0 = extract_candidates(r0['article'], a0)
print(f'Extracted {len(cands0)} candidates for row 0. Examples: {cands0[:5]}')

In [ ]:
# ── 7b. Feature engineering per candidate ──
dist_vec = CountVectorizer(binary=True, max_features=3000)
dist_vec.fit(train_df['article_clean'].tolist() + [clean_text(v) for col in ['A','B','C','D'] for v in train_df[col]])

def distractor_features(candidate, answer, article):
    from sklearn.metrics.pairwise import cosine_similarity as cs
    cand_vec = dist_vec.transform([candidate])
    ans_vec  = dist_vec.transform([clean_text(answer)])
    art_vec2 = dist_vec.transform([clean_text(article)])

    cosine_ans  = cs(cand_vec, ans_vec)[0,0]
    cosine_art  = cs(cand_vec, art_vec2)[0,0]

    art_tokens = clean_text(article).split()
    passage_freq = art_tokens.count(candidate.split()[0]) / max(len(art_tokens), 1)

    char_match = sum(1 for a,b in zip(candidate, clean_text(answer)) if a==b) / max(len(clean_text(answer)),1)
    length_ratio = len(candidate.split()) / max(len(clean_text(answer).split()), 1)

    return [cosine_ans, cosine_art, passage_freq, char_match, length_ratio]

print('Distractor feature function defined.')
print('Test features:', distractor_features(cands0[0], a0, r0['article']))

In [ ]:
# ── 7c. Build training data for distractor ranker ──
print('Building distractor ranker training data (sampling rows)...')
DIST_SAMPLE = min(500, len(train_df))
dist_X, dist_y = [], []

for _, row in train_df.sample(DIST_SAMPLE, random_state=42).iterrows():
    correct_ans = row[row['answer']]
    distractors = [row[o] for o in ['A','B','C','D'] if o != row['answer']]
    article = row['article']

    candidates = extract_candidates(article, correct_ans, max_ngram=2)
    if len(candidates) < 3:
        continue

    for cand in candidates[:30]:  # limit to 30 candidates per row
        feats = distractor_features(cand, correct_ans, article)
        # Label 1 if this candidate matches one of the actual distractors
        label = int(any(cand in clean_text(d) or clean_text(d) in cand for d in distractors))
        dist_X.append(feats)
        dist_y.append(label)

dist_X = np.array(dist_X); dist_y = np.array(dist_y)
print(f'Distractor training set: {dist_X.shape}, positive rate: {dist_y.mean():.3f}')

In [ ]:
# Train distractor ranker
dist_ranker = LogisticRegression(max_iter=500, C=1.0)
dist_ranker.fit(dist_X, dist_y)
dist_train_acc = accuracy_score(dist_y, dist_ranker.predict(dist_X))
print(f'Distractor ranker train accuracy: {dist_train_acc:.4f}')

joblib.dump(dist_ranker, f'{MODELS_B_DIR}/distractor_ranker.pkl')
joblib.dump(dist_vec,    f'{MODELS_B_DIR}/vectorizer_b.pkl')
print('Distractor ranker saved.')

In [ ]:
# ── Evaluate on val set: Precision@3, Recall@3, F1@3 ──
print('Evaluating distractor ranker on validation set...')
VAL_EVAL = min(200, len(val_df))
p3_scores, r3_scores = [], []

for _, row in val_df.sample(VAL_EVAL, random_state=42).iterrows():
    correct_ans = row[row['answer']]
    gold_distractors = {clean_text(row[o]) for o in ['A','B','C','D'] if o != row['answer']}
    candidates = extract_candidates(row['article'], correct_ans, max_ngram=2)
    if len(candidates) < 3:
        continue

    feats = [distractor_features(c, correct_ans, row['article']) for c in candidates[:50]]
    scores = dist_ranker.predict_proba(feats)[:,1]

    # Diversity penalty
    ranked = [candidates[i] for i in np.argsort(scores)[::-1]]
    selected = []
    for cand in ranked:
        if len(selected) == 3: break
        if not selected:
            selected.append(cand)
        else:
            cand_vec = dist_vec.transform([cand])
            too_similar = any(
                cosine_similarity(cand_vec, dist_vec.transform([s]))[0,0] > 0.8
                for s in selected
            )
            if not too_similar:
                selected.append(cand)

    if len(selected) < 3:
        selected += ranked[:3-len(selected)]
    selected = selected[:3]

    sel_set = set(selected)
    tp = len(sel_set & gold_distractors)
    p3_scores.append(tp / 3)
    r3_scores.append(tp / max(len(gold_distractors), 1))

dist_p3 = np.mean(p3_scores)
dist_r3 = np.mean(r3_scores)
dist_f3 = 2 * dist_p3 * dist_r3 / max(dist_p3 + dist_r3, 1e-9)
print(f'Distractor Ranker — Precision@3: {dist_p3:.4f}, Recall@3: {dist_r3:.4f}, F1@3: {dist_f3:.4f}')

In [ ]:
# Confusion Matrix for distractor ranker on labelled dataset
from sklearn.model_selection import train_test_split as tts
X_dr_train, X_dr_val, y_dr_train, y_dr_val = tts(dist_X, dist_y, test_size=0.2, random_state=42)
dist_ranker2 = LogisticRegression(max_iter=500).fit(X_dr_train, y_dr_train)
dr_pred = dist_ranker2.predict(X_dr_val)
cm_dist = confusion_matrix(y_dr_val, dr_pred)

fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm_dist, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not Distractor','Distractor'], yticklabels=['Not Distractor','Distractor'])
ax.set_title('Distractor Ranker Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/10_distractor_cm.png', dpi=100)
plt.show()
dist_ranker_acc = accuracy_score(y_dr_val, dr_pred)
print(f'Distractor Ranker Accuracy (hold-out): {dist_ranker_acc:.4f}')

### 7d. Frequency-Based Fallback Distractor Generation

In [ ]:
def freq_fallback_distractors(article, correct_answer, n=3):
    tokens = clean_text(article).split()
    freq = Counter(tokens)
    ans_tokens = set(clean_text(correct_answer).split())
    # High-freq tokens that are NOT in the answer
    stop_words = {'the','a','an','is','was','are','were','of','in','to','and','or','it',
                  'for','on','with','at','by','as','he','she','they','we','this','that'}
    candidates = [
        tok for tok, cnt in freq.most_common(100)
        if tok not in ans_tokens and tok not in stop_words and len(tok) > 3
    ]
    return candidates[:n]

# Test
fb = freq_fallback_distractors(r0['article'], a0)
print('Frequency-based fallback distractors:', fb)

### 7e. Extractive Hint Generator

In [ ]:
# Score sentences by keyword overlap with question
def get_hints(article, question, n=3):
    sentences = [s.strip() for s in re.split(r'[.!?]', article) if len(s.strip()) > 15]
    q_toks = set(clean_text(question).split())
    stop = {'the','a','an','is','was','are','were','of','in','to','and','or'}
    q_toks -= stop
    scored = []
    for i, s in enumerate(sentences):
        s_toks = set(clean_text(s).split()) - stop
        score = len(q_toks & s_toks) / max(len(q_toks), 1)
        scored.append((score, i, s))
    scored.sort(key=lambda x: x[0])
    # Hint 1 = least overlap (general), Hint 3 = most overlap (specific)
    result = [s for _, _, s in scored]
    step = max(len(result)//3, 1)
    hints = [result[0], result[min(step, len(result)-1)], result[min(2*step, len(result)-1)]]
    # Ensure 3 unique non-empty hints
    seen = set()
    unique = []
    for h in hints:
        if h and h not in seen:
            seen.add(h)
            unique.append(h)
    while len(unique) < 3:
        unique.append('(Refer to the passage for more context.)')
    return unique[:3]

test_hints = get_hints(r0['article'], r0['question'])
print('Hint 1 (general)  :', test_hints[0][:100])
print('Hint 2 (mid)      :', test_hints[1][:100])
print('Hint 3 (specific) :', test_hints[2][:100])

In [ ]:
# ── Train LogisticRegression hint scorer ──
HINT_SAMPLE = min(500, len(train_df))
hint_X, hint_y = [], []

for _, row in train_df.sample(HINT_SAMPLE, random_state=42).iterrows():
    article = row['article']
    question = row['question']
    correct_ans = row[row['answer']]
    sentences = [s.strip() for s in re.split(r'[.!?]', article) if len(s.strip()) > 15]
    ans_toks = set(clean_text(correct_ans).split())
    q_toks   = set(clean_text(question).split())

    for pos, sent in enumerate(sentences[:15]):
        s_toks = set(clean_text(sent).split())
        kw_overlap  = len(q_toks & s_toks) / max(len(q_toks), 1)
        ans_overlap = len(ans_toks & s_toks) / max(len(ans_toks), 1)
        pos_norm    = pos / max(len(sentences), 1)
        sent_len    = len(sent.split())
        hint_X.append([kw_overlap, ans_overlap, pos_norm, sent_len])
        hint_y.append(1 if ans_overlap > 0.3 else 0)

hint_X = np.array(hint_X); hint_y = np.array(hint_y)
hint_scorer = LogisticRegression(max_iter=500)
hint_scorer.fit(hint_X, hint_y)
print(f'Hint scorer trained. Train acc: {accuracy_score(hint_y, hint_scorer.predict(hint_X)):.4f}')

# R² score for regression interpretation
from sklearn.linear_model import LinearRegression
hint_reg = LinearRegression()
hint_reg.fit(hint_X, hint_y.astype(float))
hint_r2 = r2_score(hint_y, hint_reg.predict(hint_X))
print(f'Hint scorer R²: {hint_r2:.4f}')

# Precision@1
hint_p1_scores = []
HINT_EVAL = min(200, len(val_df))
for _, row in val_df.sample(HINT_EVAL, random_state=42).iterrows():
    sentences = [s.strip() for s in re.split(r'[.!?]', row['article']) if len(s.strip())>15]
    if not sentences: continue
    q_toks  = set(clean_text(row['question']).split())
    ans_toks= set(clean_text(row[row['answer']]).split())
    feats = []
    for pos, sent in enumerate(sentences[:15]):
        s_toks = set(clean_text(sent).split())
        feats.append([len(q_toks&s_toks)/max(len(q_toks),1),
                      len(ans_toks&s_toks)/max(len(ans_toks),1),
                      pos/max(len(sentences),1), len(sent.split())])
    scores = hint_scorer.predict_proba(feats)[:,1]
    best_sent = sentences[np.argmax(scores)]
    gold_overlap = len(set(clean_text(best_sent).split()) & ans_toks) / max(len(ans_toks),1)
    hint_p1_scores.append(1 if gold_overlap > 0.2 else 0)

hint_p1 = np.mean(hint_p1_scores)
print(f'Hint scorer Precision@1: {hint_p1:.4f}')

joblib.dump(hint_scorer, f'{MODELS_B_DIR}/hint_scorer.pkl')
print('Hint scorer saved.')

## Section 8 — Final Evaluation Summary

In [ ]:
# ══════════════════════════════════════════════════════
# FINAL RESULTS TABLE
# ══════════════════════════════════════════════════════

# ── Evaluate all Model A classifiers on TEST set ──
print('Evaluating on TEST set...')

lr_test_pred  = lr_model.predict(X_test)
lr_test_proba = lr_model.predict_proba(X_test)[:,1]
svm_test_pred  = svm_model.predict(X_test)
svm_test_proba = svm_model.predict_proba(X_test)[:,1]
rf_test_pred   = rf_model.predict(X_lex_test)
rf_test_proba  = rf_model.predict_proba(X_lex_test)[:,1]
ens_test_proba = (lr_test_proba + svm_test_proba + rf_test_proba) / 3
ens_test_pred  = (ens_test_proba >= 0.5).astype(int)

test_results = {
    'Logistic Regression': {
        'acc': accuracy_score(y_test, lr_test_pred),
        'f1':  f1_score(y_test, lr_test_pred, average='macro'),
        'em':  exact_match(y_test, lr_test_proba)
    },
    'SVM (Calibrated)': {
        'acc': accuracy_score(y_test, svm_test_pred),
        'f1':  f1_score(y_test, svm_test_pred, average='macro'),
        'em':  exact_match(y_test, svm_test_proba)
    },
    'Random Forest': {
        'acc': accuracy_score(y_test, rf_test_pred),
        'f1':  f1_score(y_test, rf_test_pred, average='macro'),
        'em':  exact_match(y_test, rf_test_proba)
    },
    'Ensemble (Soft Vote)': {
        'acc': accuracy_score(y_test, ens_test_pred),
        'f1':  f1_score(y_test, ens_test_pred, average='macro'),
        'em':  exact_match(y_test, ens_test_proba)
    },
}

print('\n=== MODEL A — ANSWER VERIFICATION ===')
print(f"{'Model':<25} | {'Val Acc':>7} | {'Val F1':>6} | {'Val EM':>6} | {'Test Acc':>8} | {'Test F1':>7} | {'Test EM':>7}")
print('-'*80)
val_res = results
for mname, tr in test_results.items():
    vr = val_res.get(mname, {'accuracy':0,'f1':0,'em':0})
    ve = vr['em'] if isinstance(vr['em'], float) else 0.0
    print(f"{mname:<25} | {vr['accuracy']:>7.4f} | {vr['f1']:>6.4f} | {ve:>6.4f} | {tr['acc']:>8.4f} | {tr['f1']:>7.4f} | {tr['em']:>7.4f}")

In [ ]:
print('\n=== MODEL B — DISTRACTOR GENERATION ===')
print(f"{'Metric':<20} | {'Value':>8}")
print('-'*35)
print(f"{'Precision@3':<20} | {dist_p3:>8.4f}")
print(f"{'Recall@3':<20} | {dist_r3:>8.4f}")
print(f"{'F1@3':<20} | {dist_f3:>8.4f}")
print(f"{'Ranker Accuracy':<20} | {dist_ranker_acc:>8.4f}")

print('\n=== MODEL B — HINT EXTRACTION ===')
print(f"{'Metric':<20} | {'Value':>8}")
print('-'*35)
print(f"{'Precision@1':<20} | {hint_p1:>8.4f}")
print(f"{'R² Score':<20} | {hint_r2:>8.4f}")

print('\n=== UNSUPERVISED / SEMI-SUPERVISED ===')
print(f"{'Metric':<30} | {'Value':>8}")
print('-'*45)
print(f"{'K-Means Silhouette Score':<30} | {sil_score:>8.4f}")
print(f"{'K-Means Purity':<30} | {purity:>8.4f}")
print(f"{'GMM Silhouette Score':<30} | {sil_gmm:>8.4f}")
print(f"{'GMM Log-Likelihood':<30} | {log_lik:>8.4f}")
print(f"{'Label Propagation F1 (unlabeled)':<30} | {lp_f1:>8.4f}")

## Section 9 — Export src/ Scripts

In [ ]:
%%writefile ../src/evaluate.py
"""evaluate.py — Metric computation utilities."""
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, r2_score


def compute_metrics(y_true, y_pred, y_proba=None, n_options=4):
    metrics = {
        'accuracy':  accuracy_score(y_true, y_pred),
        'macro_f1':  f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall':    recall_score(y_true, y_pred, average='macro', zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
    }
    if y_proba is not None:
        n = len(y_true) // n_options
        correct = sum(
            np.argmax(y_proba[i*n_options:(i+1)*n_options]) == np.argmax(y_true[i*n_options:(i+1)*n_options])
            for i in range(n)
            if y_true[i*n_options:(i+1)*n_options].sum() > 0
        )
        metrics['exact_match'] = correct / max(n, 1)
    return metrics


In [ ]:
%%writefile ../src/model_a_train.py
"""model_a_train.py — Train all Model A classifiers."""
import os, sys, time, warnings
import numpy as np
import joblib
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.dirname(__file__))
from preprocessing import load_features
from evaluate import compute_metrics

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from scipy.sparse import load_npz

PROCESSED = 'data/processed'
MODELS_A  = 'models/model_a/traditional'


def main():
    X_tr, X_va, X_te, y_tr, y_va, y_te = load_features(PROCESSED)
    # Lexical cols are last 5
    X_lex_tr = X_tr[:, -5:].toarray()
    X_lex_va = X_va[:, -5:].toarray()
    X_lex_te = X_te[:, -5:].toarray()

    classifiers = [
        ('lr',  LogisticRegression(max_iter=1000, C=1.0, solver='saga', n_jobs=-1), X_tr, X_va),
        ('svm', CalibratedClassifierCV(LinearSVC(max_iter=2000), cv=3), X_tr, X_va),
        ('rf',  RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1), X_lex_tr, X_lex_va),
    ]
    for name, clf, Xtr, Xva in classifiers:
        t0 = time.time()
        clf.fit(Xtr, y_tr)
        elapsed = time.time() - t0
        preds = clf.predict(Xva)
        m = compute_metrics(y_va, preds)
        print(f'{name}: acc={m["accuracy"]:.4f}, f1={m["macro_f1"]:.4f}, time={elapsed:.1f}s')
        joblib.dump(clf, os.path.join(MODELS_A, f'{name}_model.pkl'))
    print('All Model A classifiers trained and saved.')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile ../src/model_b_train.py
"""model_b_train.py — Train Model B (distractor ranker + hint scorer)."""
import os, sys, re, string, warnings
import numpy as np
import pandas as pd
import joblib
warnings.filterwarnings('ignore')

from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROCESSED = 'data/processed'
MODELS_B  = 'models/model_b/traditional'


def clean_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', text).strip()


def extract_candidates(article, answer, max_ngram=2):
    tokens = clean_text(article).split()
    ans_clean = clean_text(answer)
    cands = set()
    for n in range(1, max_ngram+1):
        for i in range(len(tokens)-n+1):
            p = ' '.join(tokens[i:i+n])
            if p != ans_clean and len(p) > 2:
                cands.add(p)
    return list(cands)


def main():
    train_df = pd.read_csv(os.path.join('data/raw', 'train.csv'))
    # Build vocab
    dist_vec = CountVectorizer(binary=True, max_features=3000)
    dist_vec.fit(train_df['article'].apply(clean_text).tolist())
    joblib.dump(dist_vec, os.path.join(MODELS_B, 'vectorizer_b.pkl'))

    SAMPLE = min(500, len(train_df))
    dist_X, dist_y = [], []
    for _, row in train_df.sample(SAMPLE, random_state=42).iterrows():
        correct_ans = row[row['answer']]
        distractors = [row[o] for o in ['A','B','C','D'] if o != row['answer']]
        cands = extract_candidates(row['article'], correct_ans)
        if len(cands) < 3: continue
        for cand in cands[:30]:
            cv = dist_vec.transform([cand])
            av = dist_vec.transform([clean_text(correct_ans)])
            cos = cosine_similarity(cv, av)[0,0]
            freq = clean_text(row['article']).split().count(cand.split()[0]) / max(len(clean_text(row['article']).split()),1)
            char = sum(1 for a,b in zip(cand, clean_text(correct_ans)) if a==b)/max(len(clean_text(correct_ans)),1)
            lr = len(cand.split())/max(len(clean_text(correct_ans).split()),1)
            cart = cosine_similarity(cv, dist_vec.transform([clean_text(row['article'])]))[0,0]
            label = int(any(cand in clean_text(d) or clean_text(d) in cand for d in distractors))
            dist_X.append([cos, cart, freq, char, lr])
            dist_y.append(label)
    dist_X = np.array(dist_X); dist_y = np.array(dist_y)
    ranker = LogisticRegression(max_iter=500).fit(dist_X, dist_y)
    joblib.dump(ranker, os.path.join(MODELS_B, 'distractor_ranker.pkl'))
    print(f'Distractor ranker trained. Acc: {accuracy_score(dist_y, ranker.predict(dist_X)):.4f}')

    hint_X, hint_y = [], []
    for _, row in train_df.sample(SAMPLE, random_state=42).iterrows():
        sents = [s.strip() for s in re.split(r'[.!?]', row['article']) if len(s.strip())>15]
        qt = set(clean_text(row['question']).split())
        at = set(clean_text(row[row['answer']]).split())
        for pos, sent in enumerate(sents[:15]):
            st = set(clean_text(sent).split())
            hint_X.append([len(qt&st)/max(len(qt),1), len(at&st)/max(len(at),1), pos/max(len(sents),1), len(sent.split())])
            hint_y.append(1 if len(at&st)/max(len(at),1) > 0.3 else 0)
    hint_X = np.array(hint_X); hint_y = np.array(hint_y)
    hint_scorer = LogisticRegression(max_iter=500).fit(hint_X, hint_y)
    joblib.dump(hint_scorer, os.path.join(MODELS_B, 'hint_scorer.pkl'))
    print('Hint scorer trained and saved.')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile ../src/inference.py
"""inference.py — Unified inference API."""
import os, re, string, time, warnings
import numpy as np
import joblib
warnings.filterwarnings('ignore')

from scipy.sparse import hstack, csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

_MODELS = {}

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
MODELS_A = os.path.join(BASE_DIR, '..', 'models', 'model_a', 'traditional')
MODELS_B = os.path.join(BASE_DIR, '..', 'models', 'model_b', 'traditional')


def _load_models():
    if _MODELS: return
    _MODELS['ohe']      = joblib.load(os.path.join(MODELS_A, 'ohe_vectorizer.pkl'))
    _MODELS['lr']       = joblib.load(os.path.join(MODELS_A, 'lr_model.pkl'))
    _MODELS['svm']      = joblib.load(os.path.join(MODELS_A, 'svm_model.pkl'))
    _MODELS['rf']       = joblib.load(os.path.join(MODELS_A, 'rf_model.pkl'))
    _MODELS['dist_vec'] = joblib.load(os.path.join(MODELS_B, 'vectorizer_b.pkl'))
    _MODELS['dist_rk']  = joblib.load(os.path.join(MODELS_B, 'distractor_ranker.pkl'))
    _MODELS['hint_sk']  = joblib.load(os.path.join(MODELS_B, 'hint_scorer.pkl'))


def _clean(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', text).strip()


def predict_answer(article, question, options):
    """Returns predicted answer letter ('A','B','C','D')."""
    _load_models()
    ohe = _MODELS['ohe']; lr = _MODELS['lr']; svm = _MODELS['svm']; rf = _MODELS['rf']
    arts = _clean(article); qs = _clean(question)
    texts, lex_rows = [], []
    for opt in options:
        os_c = _clean(opt)
        texts.append(f'{arts} [SEP] {qs} [SEP] {os_c}')
        at = set(arts.split()); qt = set(qs.split()); ot = set(os_c.split())
        ol = os_c.split()
        pos = arts.find(ol[0]) / max(len(arts),1) if ol and ol[0] in arts else 0.0
        lex_rows.append([len(ol), len(qs.split()), len(qt&ot), len(ot&at), pos])
    X_ohe = ohe.transform(texts)
    art_vec = type(ohe)(binary=True, vocabulary=ohe.vocabulary_)
    art_m = art_vec.transform([arts]*4)
    opt_m = art_vec.transform([_clean(o) for o in options])
    cos_f = csr_matrix(np.array([cosine_similarity(art_m[i], opt_m[i])[0,0] for i in range(4)]).reshape(-1,1))
    lex_f = csr_matrix(np.array(lex_rows, dtype=np.float32))
    X = hstack([X_ohe, cos_f, lex_f])
    lr_p  = lr.predict_proba(X)[:,1]
    svm_p = svm.predict_proba(X)[:,1]
    rf_p  = rf.predict_proba(lex_f.toarray())[:,1]
    scores = (lr_p + svm_p + rf_p) / 3
    return ['A','B','C','D'][int(np.argmax(scores))]


def generate_distractors(article, question, answer, n=3):
    """Returns list of 3 distractor strings."""
    _load_models()
    dvec = _MODELS['dist_vec']; drk = _MODELS['dist_rk']
    tokens = _clean(article).split()
    ans_c = _clean(answer)
    cands = list({' '.join(tokens[i:i+n_]) for n_ in range(1,3) for i in range(len(tokens)-n_+1)
                  if ' '.join(tokens[i:i+n_]) != ans_c and len(' '.join(tokens[i:i+n_])) > 2})
    if len(cands) < n:
        freq = Counter(tokens)
        stop = {'the','a','an','is','was','are','were','of','in','to','and','or','it'}
        cands += [t for t,_ in freq.most_common(50) if t not in _clean(answer).split() and t not in stop][:n]
    feats = []
    for cand in cands[:80]:
        cv = dvec.transform([cand]); av = dvec.transform([ans_c])
        arv = dvec.transform([_clean(article)])
        cos_a = cosine_similarity(cv, av)[0,0]
        cos_r = cosine_similarity(cv, arv)[0,0]
        freq_f = tokens.count(cand.split()[0]) / max(len(tokens),1)
        char_f = sum(1 for a,b in zip(cand, ans_c) if a==b) / max(len(ans_c),1)
        lr_f = len(cand.split()) / max(len(ans_c.split()),1)
        feats.append([cos_a, cos_r, freq_f, char_f, lr_f])
    scores = drk.predict_proba(feats)[:,1]
    ranked = [cands[i] for i in np.argsort(scores)[::-1]]
    selected = []
    for cand in ranked:
        if len(selected) == n: break
        if not selected:
            selected.append(cand)
        else:
            cv = dvec.transform([cand])
            if not any(cosine_similarity(cv, dvec.transform([s]))[0,0] > 0.8 for s in selected):
                selected.append(cand)
    while len(selected) < n:
        selected.append(ranked[len(selected)] if len(ranked) > len(selected) else 'N/A')
    return selected[:n]


def get_hints(article, question, n=3):
    """Returns list of 3 graduated hint strings."""
    _load_models()
    hsk = _MODELS['hint_sk']
    sents = [s.strip() for s in re.split(r'[.!?]', article) if len(s.strip()) > 15]
    if not sents:
        return ['(No hints available.)']*3
    qt = set(_clean(question).split())
    feats = []
    for pos, sent in enumerate(sents[:20]):
        st = set(_clean(sent).split())
        feats.append([len(qt&st)/max(len(qt),1), 0.0, pos/max(len(sents),1), len(sent.split())])
    scores = hsk.predict_proba(feats)[:,1]
    order = np.argsort(scores)
    step = max(len(order)//3, 1)
    hint_sents = [sents[order[0]], sents[order[min(step, len(order)-1)]], sents[order[min(2*step, len(order)-1)]]]
    seen, result = set(), []
    for h in hint_sents:
        if h not in seen: seen.add(h); result.append(h)
    while len(result) < 3:
        result.append('(Refer to the passage for more context.)')
    return result[:3]


In [ ]:
print('\nPhase 1 COMPLETE.')
print('All model artifacts saved.')
print('All src/ scripts written.')
print('\nHandoff: Run this notebook top-to-bottom, then return the executed .ipynb for Phase 2.')